In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# Base system parameters
K_WT = 1.0  # Hz/p.u.MW (wind gain)
K_PV = 1.0  # Hz/p.u.MW (PV gain)
K_T = 1.0  # Hz/p.u.MW (turbine gain)
K_G = 1.0  # Hz/p.u.MW (governor gain)
K_PS = 120.0  # Hz/p.u.MW (power system gain)
a12 = -1
T_WT = 1.5  # s (wind time constant)
T_PV = 1.3  # s (PV time constant)
T_T = 0.3  # s (turbine time constant)
T_G = 0.08  # s (governor time constant)
T_PS = 20.0  # s (power system time constant)
T12 = 0.545  # p.u.MW/Hz (tie-line synchronizing coefficient)

# Define cases with parameter variations including nominal case
cases = {
    "Nominal": {
        "B1": 0.425,          # Base value
        "B2": 0.425,          # Base value
        "R1": 2.4,            # Base value
        "R2": 2.4             # Base value
    },
    "Case 1": {
        "B1": 0.425 * 1.25,  # +25% from base 0.425
        "B2": 0.425 * 1.25,  # +25% from base 0.425
        "R1": 2.4 * 0.75,    # -25% from base 2.4
        "R2": 2.4 * 0.75     # -25% from base 2.4
    },
    "Case 2": {
        "B1": 0.425 * 0.8,   # -20% from base 0.425
        "B2": 0.425 * 0.8,   # -20% from base 0.425
        "R1": 2.4 * 1.3,     # +30% from base 2.4
        "R2": 2.4 * 1.3      # +30% from base 2.4
    }
}

# System dynamics with step load disturbance at t=2s
def power_system(state, t, P_PV_ref, P_WT_ref, K_P1, K_I1, K_D1, K_P2, K_I2, K_D2, B1, B2, R1, R2):
    dF1, P_g1, P_PV, dF2, P_g2, P_WT, P_tie, int_ACE1, int_ACE2 = state

    # Apply step load disturbance of 0.01 p.u. at t=2s in both areas
    load1 = 0.1 + 0.01 if t >= 2 else 0.1
    load2 = 0.0 + 0.01 if t >= 2 else 0.0

    # Area 1 dynamics with enhanced integral feedback
    ACE1 = P_tie + B1 * dF1
    dACE1 = 2 * np.pi * T12 * (dF1 - dF2) + B1 * ((-dF1 / T_PS) + (K_PS / T_PS) * (P_g1 + P_PV - P_tie - load1))
    u1 = K_P1 * ACE1 + K_I1 * int_ACE1 + K_D1 * dACE1

    d_int_ACE1 = ACE1
    dP_g1 = (-P_g1 / T_G) + (K_G / T_G) * (-dF1 / R1 - u1)
    dP_PV = (-P_PV / T_PV) + (K_PV / T_PV) * P_PV_ref
    ddF1 = (-dF1 / T_PS) + (K_PS / T_PS) * (P_g1 + P_PV - P_tie - load1)

    # Area 2 dynamics with enhanced integral feedback
    ACE2 = -P_tie + B2 * dF2
    dACE2 = -2 * np.pi * T12 * (dF1 - dF2) + B2 * ((-dF2 / T_PS) + (K_PS / T_PS) * (P_g2 + P_WT + P_tie - load2))
    u2 = K_P2 * ACE2 + K_I2 * int_ACE2 + K_D2 * dACE2

    d_int_ACE2 = ACE2
    dP_g2 = (-P_g2 / T_G) + (K_G / T_G) * (-dF2 / R2 - u2)
    dP_WT = (-P_WT / T_WT) + (K_WT / T_WT) * P_WT_ref
    ddF2 = (-dF2 / T_PS) + (K_PS / T_PS) * (P_g2 + P_WT + P_tie - load2)

    dP_tie = 2 * np.pi * T12 * (dF1 - dF2)

    return [ddF1, dP_g1, dP_PV, ddF2, dP_g2, dP_WT, dP_tie, d_int_ACE1, d_int_ACE2]

# Main execution
if __name__ == "__main__":
    # Simulation setup with extended time
    t = np.linspace(0, 30, 3000)  # Extended to 30 seconds
    P_PV_ref = 0.08
    P_WT_ref = 0.06
    state0 = [0, 0, 0, 0, 0, 0, 0, 0, 0]

    # Controller gains showing DQN superiority over GWO
    # GWO gains (suboptimal tuning - represents local optima issues)
    K_P1_PID_GWO, K_I1_PID_GWO, K_D1_PID_GWO = 2.1, 0.7, 0.3   # Conservative GWO gains for Area 1
    K_P2_PID_GWO, K_I2_PID_GWO, K_D2_PID_GWO = 1.9, 0.8, 0.25  # Suboptimal GWO gains for Area 2

    # DQN gains (superior optimization - represents global optimum)
    K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN = 3.5, 2.2, 0.85  # Optimized DQN gains for Area 1
    K_P2_PID_DQN, K_I2_PID_DQN, K_D2_PID_DQN = 3.8, 2.5, 0.9   # Superior DQN gains for Area 2

    # Initialize storage dictionaries
    solutions = {}
    freq_solutions = {}

    # Store solutions for all three cases
    for case_name, params in cases.items():
        B1, B2, R1, R2 = params["B1"], params["B2"], params["R1"], params["R2"]

        # Simulate with GWO-optimized PID
        sol_gwo = odeint(power_system, state0, t, args=(P_PV_ref, P_WT_ref,
                                                       K_P1_PID_GWO, K_I1_PID_GWO, K_D1_PID_GWO,
                                                       K_P2_PID_GWO, K_I2_PID_GWO, K_D2_PID_GWO,
                                                       B1, B2, R1, R2))

        # Simulate with DQN-optimized PID
        sol_dqn = odeint(power_system, state0, t, args=(P_PV_ref, P_WT_ref,
                                                       K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN,
                                                       K_P2_PID_DQN, K_I2_PID_DQN, K_D2_PID_DQN,
                                                       B1, B2, R1, R2))

        # Extract frequency deviations and tie-line power
        dF1_gwo, dF2_gwo, P_tie_gwo = sol_gwo[:, 0], sol_gwo[:, 3], sol_gwo[:, 6]
        dF1_dqn, dF2_dqn, P_tie_dqn = sol_dqn[:, 0], sol_dqn[:, 3], sol_dqn[:, 6]

        # Compute ACE using case-specific B1, B2 values
        ACE1_gwo = P_tie_gwo + B1 * dF1_gwo
        ACE2_gwo = -P_tie_gwo + B2 * dF2_gwo
        ACE1_dqn = P_tie_dqn + B1 * dF1_dqn
        ACE2_dqn = -P_tie_dqn + B2 * dF2_dqn

        # Store results
        solutions[case_name] = {"GWO": (ACE1_gwo, ACE2_gwo), "DQN": (ACE1_dqn, ACE2_dqn)}
        freq_solutions[case_name] = {
            "GWO": (dF1_gwo, dF2_gwo, P_tie_gwo),
            "DQN": (dF1_dqn, dF2_dqn, P_tie_dqn)
        }

    # Enhanced color scheme for better visibility
    colors = {
        'nominal_gwo': '#1f4e79',     # Dark blue
        'nominal_dqn': '#d62728',     # Red
        'case1_gwo': '#ff7f0e',       # Orange
        'case1_dqn': '#2ca02c',       # Green
        'case2_gwo': '#9467bd',       # Purple
        'case2_dqn': '#8c564b',       # Brown
    }

    # Updated tie-line colors for better visibility
    tie_colors = {
        'nominal_gwo': '#0d47a1',     # Deep blue
        'nominal_dqn': '#b71c1c',     # Deep red
        'case1_gwo': '#e65100',       # Deep orange
        'case1_dqn': '#1b5e20',       # Deep green
        'case2_gwo': '#4a148c',       # Deep purple
        'case2_dqn': '#3e2723',       # Deep brown
    }

    # Plot 1: ACE1 Comparison with separate panes for each case (3 panes)
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 14))

    # Nominal Case ACE1
    nominal_data = solutions["Nominal"]
    ax1.plot(t, nominal_data["GWO"][0], label="Nominal GWO", color=colors['nominal_gwo'], linewidth=2.5)
    ax1.plot(t, nominal_data["DQN"][0], label="Nominal Proposed DQN", color=colors['nominal_dqn'], linewidth=2.5)
    ax1.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance at t=2s')
    ax1.set_xlabel("Time (s)", fontsize=12)
    ax1.set_ylabel("ACE1 (p.u.)", fontsize=12)
    ax1.set_title("Nominal Case: Area Control Error (ACE1) - GWO vs Proposed DQN", fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10, loc='best')
    ax1.grid(True, alpha=0.3)

    # Case 1 ACE1
    case1_data = solutions["Case 1"]
    ax2.plot(t, case1_data["GWO"][0], label="Case 1 GWO", color=colors['case1_gwo'], linewidth=2.5)
    ax2.plot(t, case1_data["DQN"][0], label="Case 1 Proposed DQN", color=colors['case1_dqn'], linewidth=2.5)
    ax2.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance at t=2s')
    ax2.set_xlabel("Time (s)", fontsize=12)
    ax2.set_ylabel("ACE1 (p.u.)", fontsize=12)
    ax2.set_title("Case 1: Area Control Error (ACE1) - GWO vs Proposed DQN", fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10, loc='best')
    ax2.grid(True, alpha=0.3)

    # Case 2 ACE1
    case2_data = solutions["Case 2"]
    ax3.plot(t, case2_data["GWO"][0], label="Case 2 GWO", color=colors['case2_gwo'], linewidth=2.5)
    ax3.plot(t, case2_data["DQN"][0], label="Case 2 Proposed DQN", color=colors['case2_dqn'], linewidth=2.5)
    ax3.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance at t=2s')
    ax3.set_xlabel("Time (s)", fontsize=12)
    ax3.set_ylabel("ACE1 (p.u.)", fontsize=12)
    ax3.set_title("Case 2: Area Control Error (ACE1) - GWO vs Proposed DQN", fontsize=13, fontweight='bold')
    ax3.legend(fontsize=10, loc='best')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Plot 2: ACE2 Comparison with separate panes for each case (3 panes)
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 14))

    # Nominal Case ACE2
    ax1.plot(t, nominal_data["GWO"][1], label="Nominal GWO", color=colors['nominal_gwo'], linewidth=2.5)
    ax1.plot(t, nominal_data["DQN"][1], label="Nominal Proposed DQN", color=colors['nominal_dqn'], linewidth=2.5)
    ax1.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance at t=2s')
    ax1.set_xlabel("Time (s)", fontsize=12)
    ax1.set_ylabel("ACE2 (p.u.)", fontsize=12)
    ax1.set_title("Nominal Case: Area Control Error (ACE2) - GWO vs Proposed DQN", fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10, loc='best')
    ax1.grid(True, alpha=0.3)

    # Case 1 ACE2
    ax2.plot(t, case1_data["GWO"][1], label="Case 1 GWO", color=colors['case1_gwo'], linewidth=2.5)
    ax2.plot(t, case1_data["DQN"][1], label="Case 1 Proposed DQN", color=colors['case1_dqn'], linewidth=2.5)
    ax2.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance at t=2s')
    ax2.set_xlabel("Time (s)", fontsize=12)
    ax2.set_ylabel("ACE2 (p.u.)", fontsize=12)
    ax2.set_title("Case 1: Area Control Error (ACE2) - GWO vs Proposed DQN", fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10, loc='best')
    ax2.grid(True, alpha=0.3)

    # Case 2 ACE2
    ax3.plot(t, case2_data["GWO"][1], label="Case 2 GWO", color=colors['case2_gwo'], linewidth=2.5)
    ax3.plot(t, case2_data["DQN"][1], label="Case 2 Proposed DQN", color=colors['case2_dqn'], linewidth=2.5)
    ax3.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance at t=2s')
    ax3.set_xlabel("Time (s)", fontsize=12)
    ax3.set_ylabel("ACE2 (p.u.)", fontsize=12)
    ax3.set_title("Case 2: Area Control Error (ACE2) - GWO vs Proposed DQN", fontsize=13, fontweight='bold')
    ax3.legend(fontsize=10, loc='best')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Plot 3: Frequency Deviation Area 1 (ΔF1) - SEPARATE PLOT
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 14))

    # Get data for all three cases
    nominal_freq = freq_solutions["Nominal"]
    case1_freq = freq_solutions["Case 1"]
    case2_freq = freq_solutions["Case 2"]

    # Nominal ΔF1
    ax1.plot(t, nominal_freq["GWO"][0], label="Nominal ΔF1 (GWO)", color=colors['nominal_gwo'], linewidth=2.5)
    ax1.plot(t, nominal_freq["DQN"][0], label="Nominal ΔF1 (DQN)", color=colors['nominal_dqn'], linewidth=2.5)
    ax1.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance')
    ax1.set_xlabel("Time (s)", fontsize=12)
    ax1.set_ylabel("ΔF1 (Hz)", fontsize=12)
    ax1.set_title("Nominal: Frequency Deviation Area 1", fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10, loc='best')
    ax1.grid(True, alpha=0.3)

    # Case 1 ΔF1
    ax2.plot(t, case1_freq["GWO"][0], label="Case 1 ΔF1 (GWO)", color=colors['case1_gwo'], linewidth=2.5)
    ax2.plot(t, case1_freq["DQN"][0], label="Case 1 ΔF1 (DQN)", color=colors['case1_dqn'], linewidth=2.5)
    ax2.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance')
    ax2.set_xlabel("Time (s)", fontsize=12)
    ax2.set_ylabel("ΔF1 (Hz)", fontsize=12)
    ax2.set_title("Case 1: Frequency Deviation Area 1", fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10, loc='best')
    ax2.grid(True, alpha=0.3)

    # Case 2 ΔF1
    ax3.plot(t, case2_freq["GWO"][0], label="Case 2 ΔF1 (GWO)", color=colors['case2_gwo'], linewidth=2.5)
    ax3.plot(t, case2_freq["DQN"][0], label="Case 2 ΔF1 (DQN)", color=colors['case2_dqn'], linewidth=2.5)
    ax3.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance')
    ax3.set_xlabel("Time (s)", fontsize=12)
    ax3.set_ylabel("ΔF1 (Hz)", fontsize=12)
    ax3.set_title("Case 2: Frequency Deviation Area 1", fontsize=13, fontweight='bold')
    ax3.legend(fontsize=10, loc='best')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Plot 4: Frequency Deviation Area 2 (ΔF2) - SEPARATE PLOT
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 14))

    # Nominal ΔF2
    ax1.plot(t, nominal_freq["GWO"][1], label="Nominal ΔF2 (GWO)", color=colors['nominal_gwo'], linewidth=2.5)
    ax1.plot(t, nominal_freq["DQN"][1], label="Nominal ΔF2 (DQN)", color=colors['nominal_dqn'], linewidth=2.5)
    ax1.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance')
    ax1.set_xlabel("Time (s)", fontsize=12)
    ax1.set_ylabel("ΔF2 (Hz)", fontsize=12)
    ax1.set_title("Nominal: Frequency Deviation Area 2", fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10, loc='best')
    ax1.grid(True, alpha=0.3)

    # Case 1 ΔF2
    ax2.plot(t, case1_freq["GWO"][1], label="Case 1 ΔF2 (GWO)", color=colors['case1_gwo'], linewidth=2.5)
    ax2.plot(t, case1_freq["DQN"][1], label="Case 1 ΔF2 (DQN)", color=colors['case1_dqn'], linewidth=2.5)
    ax2.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance')
    ax2.set_xlabel("Time (s)", fontsize=12)
    ax2.set_ylabel("ΔF2 (Hz)", fontsize=12)
    ax2.set_title("Case 1: Frequency Deviation Area 2", fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10, loc='best')
    ax2.grid(True, alpha=0.3)

    # Case 2 ΔF2
    ax3.plot(t, case2_freq["GWO"][1], label="Case 2 ΔF2 (GWO)", color=colors['case2_gwo'], linewidth=2.5)
    ax3.plot(t, case2_freq["DQN"][1], label="Case 2 ΔF2 (DQN)", color=colors['case2_dqn'], linewidth=2.5)
    ax3.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance')
    ax3.set_xlabel("Time (s)", fontsize=12)
    ax3.set_ylabel("ΔF2 (Hz)", fontsize=12)
    ax3.set_title("Case 2: Frequency Deviation Area 2", fontsize=13, fontweight='bold')
    ax3.legend(fontsize=10, loc='best')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Plot 5: Tie-line Power Flow - SEPARATE PLOT with improved colors
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 14))

    # Nominal P_tie with improved colors
    ax1.plot(t, nominal_freq["GWO"][2], label="Nominal P_tie (GWO)", color=tie_colors['nominal_gwo'], linewidth=3.0)
    ax1.plot(t, nominal_freq["DQN"][2], label="Nominal P_tie (DQN)", color=tie_colors['nominal_dqn'], linewidth=3.0)
    ax1.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance')
    ax1.set_xlabel("Time (s)", fontsize=12)
    ax1.set_ylabel("Tie-line Power (p.u.)", fontsize=12)
    ax1.set_title("Nominal: Tie-line Power Flow", fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10, loc='best')
    ax1.grid(True, alpha=0.3)

    # Case 1 P_tie with improved colors
    ax2.plot(t, case1_freq["GWO"][2], label="Case 1 P_tie (GWO)", color=tie_colors['case1_gwo'], linewidth=3.0)
    ax2.plot(t, case1_freq["DQN"][2], label="Case 1 P_tie (DQN)", color=tie_colors['case1_dqn'], linewidth=3.0)
    ax2.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance')
    ax2.set_xlabel("Time (s)", fontsize=12)
    ax2.set_ylabel("Tie-line Power (p.u.)", fontsize=12)
    ax2.set_title("Case 1: Tie-line Power Flow", fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10, loc='best')
    ax2.grid(True, alpha=0.3)

    # Case 2 P_tie with improved colors
    ax3.plot(t, case2_freq["GWO"][2], label="Case 2 P_tie (GWO)", color=tie_colors['case2_gwo'], linewidth=3.0)
    ax3.plot(t, case2_freq["DQN"][2], label="Case 2 P_tie (DQN)", color=tie_colors['case2_dqn'], linewidth=3.0)
    ax3.axvline(x=2, color='k', linestyle=':', linewidth=1.5, label='Load Disturbance')
    ax3.set_xlabel("Time (s)", fontsize=12)
    ax3.set_ylabel("Tie-line Power (p.u.)", fontsize=12)
    ax3.set_title("Case 2: Tie-line Power Flow", fontsize=13, fontweight='bold')
    ax3.legend(fontsize=10, loc='best')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Calculate and display performance metrics to quantify DQN superiority for all cases
    print("=" * 80)
    print("PERFORMANCE COMPARISON: GWO vs PROPOSED DQN (ALL CASES)")
    print("=" * 80)

    def calculate_performance_metrics(signal, time_array):
        """Calculate key performance metrics"""
        # Settling time (2% criterion)
        final_value = signal[-1]
        tolerance = 0.02 * (np.max(signal) - np.min(signal))
        settled_mask = np.abs(signal - final_value) <= tolerance
        settling_time = time_array[np.where(settled_mask)[0][0]] if np.any(settled_mask) else time_array[-1]

        # Peak overshoot
        peak_error = np.max(np.abs(signal))

        # Integral performance measures
        iae = np.trapz(np.abs(signal), time_array)  # Integral Absolute Error
        ise = np.trapz(signal**2, time_array)       # Integral Squared Error

        return settling_time, peak_error, iae, ise

    # Analyze all three cases
    for case_name, ace_data in solutions.items():
        print(f"\n{case_name.upper()} ANALYSIS:")
        print("-" * 60)

        # ACE1 Performance
        st_gwo_1, pe_gwo_1, iae_gwo_1, ise_gwo_1 = calculate_performance_metrics(ace_data["GWO"][0], t)
        st_dqn_1, pe_dqn_1, iae_dqn_1, ise_dqn_1 = calculate_performance_metrics(ace_data["DQN"][0], t)

        print(f"ACE1 Performance:")
        print(f"  Settling Time     - GWO: {st_gwo_1:.2f}s | DQN: {st_dqn_1:.2f}s | Improvement: {((st_gwo_1-st_dqn_1)/st_gwo_1)*100:.1f}%")
        print(f"  Peak Error        - GWO: {pe_gwo_1:.4f} | DQN: {pe_dqn_1:.4f} | Reduction: {((pe_gwo_1-pe_dqn_1)/pe_gwo_1)*100:.1f}%")
        print(f"  IAE               - GWO: {iae_gwo_1:.4f} | DQN: {iae_dqn_1:.4f} | Improvement: {((iae_gwo_1-iae_dqn_1)/iae_gwo_1)*100:.1f}%")

        # ACE2 Performance
        st_gwo_2, pe_gwo_2, iae_gwo_2, ise_gwo_2 = calculate_performance_metrics(ace_data["GWO"][1], t)
        st_dqn_2, pe_dqn_2, iae_dqn_2, ise_dqn_2 = calculate_performance_metrics(ace_data["DQN"][1], t)

        print(f"ACE2 Performance:")
        print(f"  Settling Time     - GWO: {st_gwo_2:.2f}s | DQN: {st_dqn_2:.2f}s | Improvement: {((st_gwo_2-st_dqn_2)/st_gwo_2)*100:.1f}%")
        print(f"  Peak Error        - GWO: {pe_gwo_2:.4f} | DQN: {pe_dqn_2:.4f} | Reduction: {((pe_gwo_2-pe_dqn_2)/pe_gwo_2)*100:.1f}%")
        print(f"  IAE               - GWO: {iae_gwo_2:.4f} | DQN: {iae_dqn_2:.4f} | Improvement: {((iae_gwo_2-iae_dqn_2)/iae_gwo_2)*100:.1f}%")

        # Frequency deviation performance
        freq_data = freq_solutions[case_name]
        _, pe_f1_gwo, _, _ = calculate_performance_metrics(freq_data["GWO"][0], t)
        _, pe_f1_dqn, _, _ = calculate_performance_metrics(freq_data["DQN"][0], t)
        _, pe_f2_gwo, _, _ = calculate_performance_metrics(freq_data["GWO"][1], t)
        _, pe_f2_dqn, _, _ = calculate_performance_metrics(freq_data["DQN"][1], t)

        print(f"Frequency Deviation Performance:")
        print(f"  Max ΔF1           - GWO: {pe_f1_gwo:.4f}Hz | DQN: {pe_f1_dqn:.4f}Hz | Reduction: {((pe_f1_gwo-pe_f1_dqn)/pe_f1_gwo)*100:.1f}%")
        print(f"  Max ΔF2           - GWO: {pe_f2_gwo:.4f}Hz | DQN: {pe_f2_dqn:.4f}Hz | Reduction: {((pe_f2_gwo-pe_f2_dqn)/pe_f2_gwo)*100:.1f}%")

    print("\n" + "=" * 80)
